### 1. Imports and project folders

In [ ]:
# Find only this project (never search the user's home directory).
import sys
from pathlib import Path
_PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "project_paths.py").is_file()), None)
if _PROJECT_ROOT is None:
    raise FileNotFoundError("Open the notebook within the project folder.")
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))
from project_paths import ROOT, RESULTS_DIR, ANALYSIS_PATH, RAW_DIR, PROCESSED_DIR, require_analysis, require_merged, output_dir

from pathlib import Path
import pandas as pd

DATASET_SLUG = "nadyinky/sephora-products-and-skincare-reviews"

BASE_DIR = ROOT

PRODUCT_FILE = "product_info.csv"
REVIEW_FILES = [
    "reviews_0-250.csv",
    "reviews_250-500.csv",
    "reviews_500-750.csv",
    "reviews_750-1250.csv",
    "reviews_1250-end.csv",
]

PRODUCT_ID = "product_id"
AUTHOR_ID = "author_id"

for folder in [RAW_DIR, PROCESSED_DIR]:
    folder.mkdir(parents=True, exist_ok=True)
missing_raw = [RAW_DIR / f for f in [PRODUCT_FILE, *REVIEW_FILES] if not (RAW_DIR / f).is_file()]
if missing_raw:
    raise FileNotFoundError("Raw CSVs are not distributed. Supply them manually in "
                            f"{RAW_DIR}; missing: {[p.name for p in missing_raw]}. "
                            "No download is attempted. C–F can use the supplied processed data.")


### 2. Load Products Table and five Reviews files

In [ ]:
products = pd.read_csv(
    RAW_DIR / PRODUCT_FILE,
    low_memory=False,
    dtype={"product_id": "string", "author_id": "string"}
)

review_frames = []
file_audit_rows = []
schema_audit_rows = []
reference_review_columns = None

for filename in REVIEW_FILES:
    df = pd.read_csv(
        RAW_DIR / filename,
        low_memory=False,
    dtype={"product_id": "string", "author_id": "string"}
    )

    original_columns = set(df.columns)
    file_audit_rows.append({
        "file_name": filename,
        "row_count": len(df),
        "column_count": len(df.columns),
        "unique_product_ids": df[PRODUCT_ID].nunique(dropna=True)
            if PRODUCT_ID in df.columns else None,
        "missing_product_id": int(df[PRODUCT_ID].isna().sum())
            if PRODUCT_ID in df.columns else None,
    })

    if reference_review_columns is None:
        reference_review_columns = original_columns

    schema_audit_rows.append({
        "file_name": filename,
        "missing_columns_vs_first_file": sorted(reference_review_columns - original_columns),
        "extra_columns_vs_first_file": sorted(original_columns - reference_review_columns),
    })

    if original_columns != reference_review_columns:
        raise ValueError(f"Review schema differs in {filename}; see schema_audit_rows.")

    df["source_file"] = filename
    review_frames.append(df)

reviews = pd.concat(review_frames, ignore_index=True, sort=False)
file_audit = pd.DataFrame(file_audit_rows)
schema_audit = pd.DataFrame(schema_audit_rows)

print("Products shape:", products.shape)
print("Combined reviews shape:", reviews.shape)
display(file_audit)
display(schema_audit)

### 3. Data Type Cleaning

In [ ]:
# 1. Remove redundant index columns from original CSV files

for df in [products, reviews]:

    unnamed_cols = [
        col for col in df.columns
        if col.startswith("Unnamed:")
    ]

    if unnamed_cols:
        df.drop(columns=unnamed_cols, inplace=True)

In [ ]:
# 2. Define columns by data type

# Products Table
product_string_cols = [
    "product_id",
    "product_name",
    "brand_id",
    "brand_name",
    "size",
    "variation_type",
    "variation_value",
    "variation_desc",
    "ingredients",
    "highlights",
    "primary_category",
    "secondary_category",
    "tertiary_category"
]

product_integer_cols = [
    "loves_count",
    "reviews",
    "child_count"
]

product_float_cols = [
    "rating",
    "price_usd",
    "value_price_usd",
    "sale_price_usd",
    "child_max_price",
    "child_min_price"
]

product_boolean_cols = [
    "limited_edition",
    "new",
    "online_only",
    "out_of_stock",
    "sephora_exclusive"
]


# Reviews Table
review_string_cols = [
    "author_id",
    "product_id",
    "review_text",
    "review_title",
    "skin_tone",
    "eye_color",
    "skin_type",
    "hair_color",
    "source_file",
    "product_name",
    "brand_name"
]

review_integer_cols = [
    "rating",
    "total_feedback_count",
    "total_neg_feedback_count",
    "total_pos_feedback_count"
]

review_float_cols = [
    "helpfulness",
    "price_usd"
]

review_boolean_cols = [
    "is_recommended"
]

review_datetime_cols = [
    "submission_time"
]

In [ ]:
# 3. Standardize string columns

missing_tokens = [
    "", "nan", "none", "null", "n/a", "na"
]


def clean_string_columns(df, columns):

    for col in columns:

        if col in df.columns:

            df[col] = (
                df[col]
                .astype("string")
                .str.strip()
            )

            df[col] = df[col].mask(
                df[col].str.casefold().isin(missing_tokens)
            )

    return df


products = clean_string_columns(
    products,
    product_string_cols
)

reviews = clean_string_columns(
    reviews,
    review_string_cols
)


In [ ]:
# 4. Convert numerical columns

conversion_issues = []


def convert_numeric_columns(df, columns, table_name, target_dtype):

    for col in columns:

        if col not in df.columns:
            continue

        original = df[col]

        # Convert to numeric
        converted = pd.to_numeric(
            original,
            errors="coerce"
        )

        # Identify invalid values
        invalid_mask = (
            original.notna()
            & converted.isna()
        )

        # Check fractional values in integer columns
        fractional_mask = pd.Series(
            False,
            index=df.index
        )

        if target_dtype == "Int64":

            fractional_mask = (
                converted.notna()
                & (converted % 1 != 0)
            )

        invalid_count = int(invalid_mask.sum())
        fractional_count = int(fractional_mask.sum())

        if invalid_count > 0 or fractional_count > 0:

            conversion_issues.append({
                "table": table_name,
                "column": col,
                "invalid_values": invalid_count,
                "fractional_values_in_integer_column": fractional_count,
                "examples": str(
                    original[
                        invalid_mask | fractional_mask
                    ].unique()[:10]
                )
            })

        # Preserve fractional values instead of truncating them
        if target_dtype == "Int64" and fractional_count > 0:

            df[col] = converted.astype("Float64")

        else:

            df[col] = converted.astype(target_dtype)

    return df


products = convert_numeric_columns(
    products,
    product_integer_cols,
    "products",
    "Int64"
)

products = convert_numeric_columns(
    products,
    product_float_cols,
    "products",
    "Float64"
)

reviews = convert_numeric_columns(
    reviews,
    review_integer_cols,
    "reviews",
    "Int64"
)

reviews = convert_numeric_columns(
    reviews,
    review_float_cols,
    "reviews",
    "Float64"
)

In [ ]:
# 5. Convert Boolean columns

def convert_boolean_columns(df, columns, table_name):

    boolean_mapping = {
        "1": True,
        "0": False,
        "1.0": True,
        "0.0": False,
        "true": True,
        "false": False
    }

    for col in columns:

        if col not in df.columns:
            continue

        original = df[col]

        normalized = (
            original
            .astype("string")
            .str.strip()
            .str.casefold()
        )

        converted = normalized.map(boolean_mapping)

        # Identify unexpected values
        invalid_mask = (
            original.notna()
            & converted.isna()
        )

        if invalid_mask.any():

            conversion_issues.append({
                "table": table_name,
                "column": col,
                "invalid_values": int(invalid_mask.sum()),
                "fractional_values_in_integer_column": 0,
                "examples": str(
                    original[invalid_mask].unique()[:10]
                )
            })

        df[col] = converted.astype("boolean")

    return df


products = convert_boolean_columns(
    products,
    product_boolean_cols,
    "products"
)

reviews = convert_boolean_columns(
    reviews,
    review_boolean_cols,
    "reviews"
)

In [ ]:
# 6. Convert Datetime columns

for col in review_datetime_cols:

    if col not in reviews.columns:
        continue

    original = reviews[col]

    converted = pd.to_datetime(
        original,
        format="%Y-%m-%d",
        errors="coerce"
    )

    # Identify invalid date values
    invalid_mask = (
        original.notna()
        & converted.isna()
    )

    if invalid_mask.any():

        conversion_issues.append({
            "table": "reviews",
            "column": col,
            "invalid_values": int(invalid_mask.sum()),
            "fractional_values_in_integer_column": 0,
            "examples": str(
                original[invalid_mask].unique()[:10]
            )
        })

    reviews[col] = converted

In [ ]:
# 7. Generate datatype audit

dtype_audit = pd.concat([

    pd.DataFrame({
        "table": "products",
        "column": products.columns,
        "dtype": products.dtypes.astype(str).values,
        "missing_values": products.isna().sum().values
    }),

    pd.DataFrame({
        "table": "reviews",
        "column": reviews.columns,
        "dtype": reviews.dtypes.astype(str).values,
        "missing_values": reviews.isna().sum().values
    })

], ignore_index=True)

In [ ]:
# 8. Generate conversion issue report

conversion_issue_audit = pd.DataFrame(
    conversion_issues,
    columns=[
        "table",
        "column",
        "invalid_values",
        "fractional_values_in_integer_column",
        "examples"
    ]
)

In [ ]:
# 9. Display audit reports

print("Datatype Audit:")
display(dtype_audit)

print("\nProducts shape:", products.shape)
print("Reviews shape:", reviews.shape)

print("Conversion Issues:")
display(conversion_issue_audit)

### 4. Structural checks and duplicate audit

In [ ]:
# 1. Check required columns

for table_name, df in [("products", products), ("reviews", reviews)]:
    if PRODUCT_ID not in df.columns:
        raise KeyError(f"{PRODUCT_ID} is missing from {table_name} table.")


# 2. Check missing and duplicate Product IDs

product_id_missing = int(products[PRODUCT_ID].isna().sum())

duplicate_product_id_rows = int(
    products[PRODUCT_ID].duplicated(keep=False).sum()
)

review_product_id_missing = int(reviews[PRODUCT_ID].isna().sum())


# 3. Detect duplicate reviews
# Ignore feedback counts and source_file when comparing records

excluded_columns = [
    "Unnamed: 0",
    "helpfulness", 
    "total_feedback_count",
    "total_neg_feedback_count",
    "total_pos_feedback_count",
    "source_file"
]

duplicate_subset = [
    col for col in reviews.columns
    if col not in excluded_columns
]

# Count all rows belonging to duplicate groups
duplicate_mask = reviews.duplicated(
    subset=duplicate_subset,
    keep=False
)

duplicate_review_rows = int(duplicate_mask.sum())

# Count actual rows that will be removed
duplicate_rows_removed = int(
    reviews.duplicated(
        subset=duplicate_subset,
        keep="first"
    ).sum()
)

print("Duplicate review rows identified:", duplicate_review_rows)
print("Duplicate rows to be removed:", duplicate_rows_removed)


# 4. Remove duplicate reviews, keeping the first occurrence

reviews = reviews.drop_duplicates(
    subset=duplicate_subset,
    keep="first"
).copy()

print("Reviews after deduplication:", len(reviews))


# 5. Identify repeated author-product pairs AFTER deduplication

repeated_pairs = pd.DataFrame()

if AUTHOR_ID in reviews.columns:

    pair_counts = (
        reviews.dropna(subset=[AUTHOR_ID, PRODUCT_ID])
        .groupby([AUTHOR_ID, PRODUCT_ID])
        .size()
        .rename("number_of_reviews")
        .reset_index()
    )

    repeated_pairs = pair_counts[
        pair_counts["number_of_reviews"] > 1
    ].sort_values(
        "number_of_reviews",
        ascending=False
    )

repeated_author_product_pair_count = len(repeated_pairs)

rows_in_repeated_pairs = (
    int(repeated_pairs["number_of_reviews"].sum())
    if len(repeated_pairs) else 0
)


# 6. Generate duplicate audit summary

duplicate_audit = pd.DataFrame([
    {"check": "product_rows", "value": len(products)},
    {"check": "product_columns", "value": len(products.columns)},
    {"check": "original_review_rows", "value": len(reviews) + duplicate_rows_removed},
    {"check": "review_rows_after_deduplication", "value": len(reviews)},
    {"check": "duplicate_review_rows_identified", "value": duplicate_review_rows},
    {"check": "duplicate_review_rows_removed", "value": duplicate_rows_removed},
    {"check": "missing_product_table_product_id", "value": product_id_missing},
    {"check": "duplicate_product_id_rows", "value": duplicate_product_id_rows},
    {"check": "missing_review_product_id", "value": review_product_id_missing},
    {"check": "repeated_author_product_pairs_after_deduplication",
     "value": repeated_author_product_pair_count},
    {"check": "rows_in_repeated_author_product_pairs",
     "value": rows_in_repeated_pairs},
])

display(duplicate_audit)
display(repeated_pairs.head(20))


# 7. Validate Products Table

if product_id_missing > 0:
    raise ValueError(
        "Products table has missing product_id values. Investigate before merging."
    )

if duplicate_product_id_rows > 0:
    raise ValueError(
        "Products table has duplicated product_id values. Resolve before merging."
    )

### 5. Impute Missing Values for Consistent Authors

In [ ]:
# 1. Define reviewer-level attributes

demographic_cols = [
    "skin_tone",
    "eye_color",
    "skin_type",
    "hair_color"
]

attribute_audit_rows = []
conflict_incomplete_rows = []

# Store authors eligible for imputation
incomplete_consistent_ids = {}


# 2. Check consistency for each attribute

for col in demographic_cols:

    # Prepare data for author-level analysis
    temp = pd.DataFrame({
        AUTHOR_ID: reviews[AUTHOR_ID],
        "value": reviews[col]
    })

    # Normalize values for comparison (case-insensitive)
    temp["normalized_value"] = temp["value"].str.casefold()

    # Summarize each author's records
    author_summary = (
        temp.groupby(AUTHOR_ID)
        .agg(
            total_records=("value", "size"),
            non_missing_records=("value", "count"),
            unique_values=("normalized_value", "nunique"),
            observed_values=(
                "value",
                lambda x: " | ".join(
                    pd.unique(x.dropna()).astype(str)
                )
            )
        )
        .reset_index()
    )

    # Identify missing and conflicting values
    author_summary["has_missing"] = (
        author_summary["non_missing_records"]
        < author_summary["total_records"]
    )

    author_summary["has_conflict"] = (
        author_summary["unique_values"] > 1
    )

    # Classify authors
    complete = ~author_summary["has_missing"]

    incomplete_consistent = (
        author_summary["has_missing"]
        & (author_summary["unique_values"] == 1)
    )

    incomplete_conflicting = (
        author_summary["has_missing"]
        & (author_summary["unique_values"] > 1)
    )

    all_missing = (
        author_summary["non_missing_records"] == 0
    )

    # Store eligible author IDs
    incomplete_consistent_ids[col] = set(
        author_summary.loc[incomplete_consistent, AUTHOR_ID]
    )

    # Extract authors with conflicting values and missing records
    conflict_details = author_summary[
        incomplete_conflicting
    ].copy()

    conflict_details.insert(0, "attribute", col)

    conflict_incomplete_rows.extend(
        conflict_details.to_dict("records")
    )

    # Record audit statistics
    attribute_audit_rows.append({
        "attribute": col,
        "complete_authors": int(complete.sum()),
        "incomplete_consistent_authors": int(
            incomplete_consistent.sum()
        ),
        "incomplete_conflicting_authors": int(
            incomplete_conflicting.sum()
        ),
        "all_missing_authors": int(all_missing.sum()),
        "total_authors": len(author_summary)
    })


In [ ]:
# 3. Fill missing values for consistent authors

fill_audit_rows = []

for col in demographic_cols:

    # Retrieve authors already classified as incomplete + consistent
    eligible_authors = incomplete_consistent_ids[col]

    # Record missing values before imputation
    missing_before = int(reviews[col].isna().sum())

    # Prepare their existing non-missing values
    temp = reviews[
        reviews[AUTHOR_ID].isin(eligible_authors)
    ][[AUTHOR_ID, col]].dropna(
        subset=[col]
    )

    # Create author-level mapping using the first observed value
    fill_map = (
        temp.drop_duplicates(subset=[AUTHOR_ID])
        .set_index(AUTHOR_ID)[col]
    )

    # Identify missing values belonging to eligible authors
    fill_candidates = reviews[AUTHOR_ID].map(fill_map)

    fill_mask = (
        reviews[AUTHOR_ID].isin(eligible_authors)
        & reviews[col].isna()
        & fill_candidates.notna()
    )

    # Fill missing values
    reviews.loc[fill_mask, col] = fill_candidates[fill_mask]

    filled_count = int(fill_mask.sum())
    missing_after = int(reviews[col].isna().sum())

    fill_audit_rows.append({
        "attribute": col,
        "missing_before": missing_before,
        "values_filled": filled_count,
        "missing_after": missing_after
    })

    print(f"{col}: {filled_count:,} values filled")

# 4. Generate imputation summary

fill_audit = pd.DataFrame(fill_audit_rows)

display(fill_audit)

In [ ]:
# 5. Generate audit reports

attribute_audit = pd.DataFrame(attribute_audit_rows)

conflict_incomplete_audit = pd.DataFrame(
    conflict_incomplete_rows
)

display(attribute_audit)

print("Authors with conflicting and incomplete attributes:")
display(conflict_incomplete_audit)

### 6. Check review counts per product and potential 200-review cap

In [ ]:
product_review_counts = (
    reviews.dropna(subset=[PRODUCT_ID])
    .groupby(PRODUCT_ID)
    .size()
    .rename("review_count_in_dataset")
    .reset_index()
    .sort_values("review_count_in_dataset", ascending=False)
)

if len(product_review_counts):
    max_reviews_per_product = int(product_review_counts["review_count_in_dataset"].max())
    products_exactly_200 = int((product_review_counts["review_count_in_dataset"] == 200).sum())
    products_over_200 = int((product_review_counts["review_count_in_dataset"] > 200).sum())
else:
    max_reviews_per_product = None
    products_exactly_200 = 0
    products_over_200 = 0

cap_audit = pd.DataFrame([
    {"metric": "unique_products_with_reviews", "value": len(product_review_counts)},
    {"metric": "maximum_reviews_per_product", "value": max_reviews_per_product},
    {"metric": "products_with_exactly_200_reviews", "value": products_exactly_200},
    {"metric": "products_with_more_than_200_reviews", "value": products_over_200},
])

display(cap_audit)
display(product_review_counts.head(20))

### 7. Merge Reviews with Products

In [ ]:
rename_map = {
    col: f"product_{col}"
    for col in products.columns
    if col != PRODUCT_ID and col in reviews.columns
}
products_for_merge = products.rename(columns=rename_map)

merged_reviews = reviews.merge(
    products_for_merge,
    on=PRODUCT_ID,
    how="left",
    validate="many_to_one",
    indicator=True
)

join_audit = (
    merged_reviews["_merge"]
    .value_counts(dropna=False)
    .rename_axis("merge_status")
    .reset_index(name="row_count")
)
join_audit["percentage"] = (
    join_audit["row_count"] / len(merged_reviews) * 100 if len(merged_reviews) else 0
)

merged_reviews["product_match_status"] = merged_reviews["_merge"].map({
    "both": "matched",
    "left_only": "unmatched_review_product_id",
    "right_only": "product_without_review"
}).astype("string")
merged_reviews = merged_reviews.drop(columns=["_merge"])

matched_rows = int((merged_reviews["product_match_status"] == "matched").sum())
unmatched_rows = int(
    (merged_reviews["product_match_status"] == "unmatched_review_product_id").sum()
)

display(join_audit)
print(f"Original combined review rows: {len(reviews):,}")
print(f"Merged rows: {len(merged_reviews):,}")
print(f"Matched review rows: {matched_rows:,}")
print(f"Unmatched review rows: {unmatched_rows:,}")

assert len(merged_reviews) == len(reviews), (
    "Row count changed after left join. Investigate the merge."
)

### 8. Save merged Parquet and audit reports

In [ ]:
parquet_path = PROCESSED_DIR / "merged_reviews.parquet"
if parquet_path.exists():
    raise FileExistsError(f"Refusing to overwrite {parquet_path}; choose a new SEPHORA_DATA_DIR.")
merged_reviews.to_parquet(parquet_path, index=False, engine="pyarrow")

print("Saved Parquet:", parquet_path.resolve())
# Data-quality summaries accompanying the integrated dataset.
summary_dir = output_dir("A")
summary_dir.mkdir(parents=True, exist_ok=True)
summary_names = {'file_audit': 'input_files', 'schema_audit': 'schema_summary', 'dtype_audit': 'data_types', 'conversion_issue_audit': 'conversion_issues', 'duplicate_audit': 'duplicates', 'attribute_audit': 'author_consistency', 'fill_audit': 'imputation', 'cap_audit': 'review_count_summary', 'join_audit': 'join_summary'}
for variable, filename in summary_names.items():
    globals()[variable].to_csv(summary_dir / f"{filename}.csv", index=False)


### 9. Final validation: read the Parquet file back

In [ ]:
check = pd.read_parquet(parquet_path)

print("Parquet file exists:", parquet_path.exists())
print("Parquet shape:", check.shape)
print("Merged DataFrame shape:", merged_reviews.shape)

assert check.shape == merged_reviews.shape
print("Validation passed.")